# 📊 Analítica Predictiva con Apache Spark — Superstore
### Curso de Introducción a Big Data
---
En este notebook exploraremos tres técnicas fundamentales de analítica predictiva usando el dataset de **Superstore** y **Apache Spark MLlib**:

| # | Técnica | Objetivo |
|---|---------|----------|
| 1 | **Regresión Lineal** | Predecir el valor continuo de `Profit` |
| 2 | **Regresión Logística** | Clasificar si una venta es "alta" o "baja" |
| 3 | **Series de Tiempo** | Pronosticar ventas futuras con Prophet |

> 💡 **Principio clave de Big Data:** Procesamos los datos con Spark (distribuido) antes de entrenar modelos, lo que nos permite escalar a millones de registros.


---
## 0. Librerías e Importaciones

Importamos las librerías necesarias de **Spark MLlib** (modelos distribuidos) y librerías de Python para visualización y series de tiempo.


In [0]:
# ── Spark SQL & funciones ────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import col, to_date, sum as spark_sum, avg, count, month, year

# ── Spark MLlib — feature engineering ───────────────────────
from pyspark.ml.feature import VectorAssembler, StringIndexer, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import RegressionEvaluator, BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# ── Spark MLlib — modelos ────────────────────────────────────
from pyspark.ml.regression import LinearRegression
from pyspark.ml.classification import LogisticRegression

# ── Spark MLlib — evaluación ─────────────────────────────────
from pyspark.ml.evaluation import (
    RegressionEvaluator,
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)

# ── Visualización ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import pandas as pd
import numpy as np

# ── Limpieza general de variables ─────────────────────────────
from pyspark.sql.functions import regexp_replace

print("✅ Librerías cargadas correctamente")


---
## 1. Carga y Exploración de Datos

Cargamos el CSV de Superstore directamente en un **Spark DataFrame**.  
Spark infiere automáticamente el esquema (`inferSchema=True`), lo que ahorra tiempo en datasets grandes.

> 📌 **Ajusta la ruta** al volumen o DBFS donde hayas subido el archivo.


In [0]:
# ── Ajusta la ruta según donde subiste el archivo ────────────
FILE_PATH = "/Volumes/workspace/trabajosia/info_trabajo/Superstore.csv"
# FILE_PATH = "/Volumes/<catalogo>/<schema>/<volumen>/Sample_-_Superstore.csv"

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("encoding", "ISO-8859-1")
    .csv(FILE_PATH)
)

print(f"📦 Filas: {df_raw.count():,}  |  Columnas: {len(df_raw.columns)}")
df_raw.printSchema()


In [0]:
# Vista previa de los primeros registros
display(df_raw.limit(10))


---
## 2. Limpieza y Preparación de Datos

Antes de entrenar cualquier modelo es fundamental garantizar la **calidad de los datos**.  
Realizamos las siguientes operaciones:

1. Eliminar filas con valores nulos en columnas clave.
2. Convertir `Order Date` a tipo fecha.
3. Calcular estadísticas descriptivas básicas.


In [0]:
# ── 1. Eliminar nulos en columnas críticas ───────────────────
COLS_CLAVE = ["Sales", "Quantity", "Discount", "Profit", "Category", "Sub-Category"]

df = df_raw.dropna(subset=COLS_CLAVE)

# ── Limpieza general de variables ─────────────────────────────
from pyspark.sql.functions import regexp_replace

# Quitar espacios extra y caracteres no alfanuméricos en columnas string
for col_name, dtype in df_raw.dtypes:
    if dtype == "string":
        df = df.withColumn(
            col_name,
            regexp_replace(col(col_name), r"[^a-zA-Z0-9\s\-\.,]", "")
        )
        df = df.withColumn(
            col_name,
            F.trim(col(col_name))
        )
# Quitar valores no numéricos en columnas numéricas
for num_col in ["Sales", "Quantity", "Discount", "Profit"]:
    df = df.filter(col(num_col).rlike(r"^-?\d+(\.\d+)?$"))
# Convertir tipos de variables según corresponda
df = df.withColumn("Sales", col("Sales").cast("double"))
df = df.withColumn("Quantity", col("Quantity").cast("integer"))
df = df.withColumn("Discount", col("Discount").cast("double"))
df = df.withColumn("Profit", col("Profit").cast("double"))

# ── 2. Convertir fechas ───────────────────────────────────────
df = df.withColumn("Order Date", to_date(col("Order Date"), "M/d/yyyy"))
df = df.withColumn("Ship Date",  to_date(col("Ship Date"),  "M/d/yyyy"))

# ── 3. Verificar resultado ────────────────────────────────────
print(f"✅ Filas después de limpieza: {df.count():,}")
print(f"   Rango de fechas: {df.agg(F.min('Order Date'), F.max('Order Date')).collect()[0]}")


In [0]:
# Estadísticas descriptivas de las variables numéricas clave
display(df.select("Sales", "Quantity", "Discount", "Profit").describe())


In [0]:
# ── Distribución de ventas por Categoría ─────────────────────
# Filter valid numeric strings before casting
df_clean = df.filter(col("Sales").rlike(r"^[0-9.]+$"))

ventas_cat = (
    df_clean.withColumn("Sales_num", col("Sales").cast("double"))
    .groupBy("Category")
    .agg(
        spark_sum("Sales_num").alias("Ventas_Totales"),
        spark_sum("Profit").alias("Profit_Total"),
        count("*").alias("N_Ordenes")
    )
    .orderBy("Ventas_Totales", ascending=False)
    .toPandas()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(ventas_cat["Category"], ventas_cat["Ventas_Totales"] / 1e3, color=["#4e79a7","#f28e2b","#e15759"])
axes[0].set_title("Ventas Totales por Categoría (miles $)", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Miles USD")
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x:,.0f}K"))

axes[1].bar(ventas_cat["Category"], ventas_cat["Profit_Total"] / 1e3, color=["#4e79a7","#f28e2b","#e15759"])
axes[1].set_title("Profit Total por Categoría (miles $)", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Miles USD")
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x:,.0f}K"))

plt.tight_layout()
plt.show()
print(ventas_cat.to_string(index=False))

---
## 3. Regresión Lineal — Predicción de `Profit`

### 🎯 Objetivo
Construir un modelo que **prediga numéricamente el Profit** (ganancia) de cada transacción.

### ¿Cuándo usar Regresión Lineal?
Cuando la variable que queremos predecir es **continua** (un número real) y sospechamos que existe una relación aproximadamente lineal con las variables predictoras.

### Variables del modelo

| Rol | Variable | Descripción |
|-----|----------|-------------|
| **Y (target)** | `Profit` | Ganancia por transacción |
| **X₁** | `Sales` | Monto de venta |
| **X₂** | `Quantity` | Unidades vendidas |
| **X₃** | `Discount` | Descuento aplicado (0 a 1) |

### Ecuación del modelo
$$\hat{Profit} = \beta_0 + \beta_1 \cdot Sales + \beta_2 \cdot Quantity + \beta_3 \cdot Discount + \varepsilon$$


In [0]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Convertir Spark DataFrame → Pandas
df_pd = df.select("Sales", "Quantity", "Discount", "Profit").toPandas()

X = df_pd[["Sales", "Quantity", "Discount"]]
y = df_pd["Profit"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"📊 Entrenamiento: {len(X_train):,} filas")
print(f"🧪 Prueba       : {len(X_test):,} filas")

# Modelo
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)

print(f"\n📈 R²  : {r2_score(y_test, y_pred):.4f}")
print(f"📉 RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

# ── Coeficientes del modelo ──────────────────────────────────
print("\n📌 Intercepto (β₀):", round(lr_model.intercept_, 4))
print("\n📋 Coeficientes:")

coef_df = pd.DataFrame({
    "Variable"    : X.columns,
    "Coeficiente" : lr_model.coef_.round(4)
}).sort_values("Coeficiente", ascending=False)

print(coef_df.to_string(index=False))

In [0]:
# ─ Tabla de predicciones vs reales ──────────────────
results = X_test.copy()
results["Profit_Real"]      = y_test.values
results["Profit_Predicho"]  = y_pred.round(2)
results["Error"]            = (results["Profit_Real"] - results["Profit_Predicho"]).round(2)

results.head(10)


### 🔍 Interpretación de coeficientes

| Coeficiente | Interpretación |
|-------------|---------------|
| **β₁ (Sales)** | Por cada $1 adicional en ventas, el Profit cambia en β₁ dólares |
| **β₂ (Quantity)** | Por cada unidad adicional vendida, el Profit cambia en β₂ |
| **β₃ (Discount)** | Por cada 10% adicional de descuento, el Profit cambia en β₃×0.1 |

> ⚠️ Si β₃ es negativo, confirma que los descuentos reducen la ganancia.


### 📐 ¿Qué significa cada métrica?

| Métrica | Fórmula simplificada | Interpretación |
|---------|---------------------|----------------|
| **R²** | Varianza explicada / Varianza total | Cercano a 1 = buen ajuste |
| **RMSE** | √(media de errores²) | Mismo orden que Y; penaliza errores grandes |
| **MAE** | media de \|errores\| | Más robusto a outliers que RMSE |


---
## 4. Regresión Logística — Clasificación de Ventas Altas

### 🎯 Objetivo
Predecir si una transacción tendrá **ventas altas** o **ventas bajas** — un problema de **clasificación binaria**.

### ¿Cuándo usar Regresión Logística?
Cuando la variable a predecir es **categórica binaria** (Sí/No, Alto/Bajo, 0/1).  
La regresión logística modela la **probabilidad** de pertenecer a la clase positiva.

### Definición de la variable objetivo

Definimos **"venta alta"** (HighSales = 1) cuando `Sales` supera el **percentil 75** de la distribución.

$$P(HighSales=1) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 X_1 + \beta_2 X_2)}}$$


In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
import numpy as np

# ── PASO 1: Crear variable binaria HighSales ─────────────────
p75 = df.approxQuantile("Sales", [0.75], 0.01)[0]
print(f"📊 Percentil 75 de Sales: ${p75:,.2f}")

df_logreg = df.withColumn(
    "HighSales",
    (col("Sales") > p75).cast("integer")
)

# Verificar balance de clases
balance = df_logreg.groupBy("HighSales").count().toPandas()
print("\nBalance de clases:")
print(balance.to_string(index=False))

# ── PASO 2: Spark DataFrame → Pandas ────────────────────────
df_pd_log = df_logreg.select("Quantity", "Discount", "Profit", "HighSales").toPandas()

# ── PASO 3: Definir X e y ───────────────────────────────────
X_log = df_pd_log[["Quantity", "Discount", "Profit"]]
y_log = df_pd_log["HighSales"]

# ── PASO 4: División train / test (70 / 30) ─────────────────
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(
    X_log, y_log, test_size=0.3, random_state=42
)

print(f"\n📊 Entrenamiento: {len(X_train_log):,} filas")
print(f"🧪 Prueba       : {len(X_test_log):,} filas")

# ── PASO 5: Entrenar modelo ──────────────────────────────────
log_model = LogisticRegression(max_iter=100, C=100, random_state=42)
log_model.fit(X_train_log, y_train_log)

print("\n✅ Modelo entrenado")
print(f"   Intercepto     : {log_model.intercept_[0]:.4f}")
print(f"   Coef. Quantity : {log_model.coef_[0][0]:.4f}")
print(f"   Coef. Discount : {log_model.coef_[0][1]:.4f}")
print(f"   Coef. Profit   : {log_model.coef_[0][2]:.4f}")

# ── PASO 6: Predicciones y métricas ─────────────────────────
y_pred_log = log_model.predict(X_test_log)
y_prob_log = log_model.predict_proba(X_test_log)[:, 1]

auc = roc_auc_score(y_test_log, y_prob_log)
acc = accuracy_score(y_test_log, y_pred_log)
f1  = f1_score(y_test_log, y_pred_log)

print("\n" + "=" * 40)
print("  MÉTRICAS — REGRESIÓN LOGÍSTICA")
print("=" * 40)
print(f"  AUC-ROC  : {auc:.4f}")
print(f"  Accuracy : {acc:.4f}  ({acc*100:.1f}% de acierto)")
print(f"  F1 Score : {f1:.4f}")
print("=" * 40)


### 📐 ¿Qué significan estas métricas?

| Métrica | Rango | Interpretación |
|---------|-------|----------------|
| **AUC-ROC** | 0.5 – 1.0 | 0.5 = azar; 1.0 = perfecto. Mide discriminación del modelo |
| **Accuracy** | 0 – 1 | % de predicciones correctas. ¡Cuidado con clases desbalanceadas! |
| **F1 Score** | 0 – 1 | Balance entre precisión y recall. Ideal para clases desbalanceadas |


In [0]:
# ── PASO 7: Matriz de Confusión ───────────────────────────────
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test_log, y_pred_log)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Venta Baja (0)", "Venta Alta (1)"]
)
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Matriz de Confusión — Regresión Logística", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# Métricas por clase
tn, fp, fn, tp = cm.ravel()
print(f"\nVerdaderos Positivos (TP): {tp}")
print(f"Verdaderos Negativos (TN): {tn}")
print(f"Falsos Positivos    (FP): {fp}  → clasificó como 'alta' pero era 'baja'")
print(f"Falsos Negativos    (FN): {fn}  → clasificó como 'baja' pero era 'alta'")
print(f"\nPrecisión (clase 1): {tp/(tp+fp):.3f}")
print(f"Recall    (clase 1): {tp/(tp+fn):.3f}")


### 🗺️ Interpretación de la Matriz de Confusión

```
                  Predicho
                  Baja    Alta
Real  Baja   [  TN  |  FP  ]
      Alta   [  FN  |  TP  ]
```
- **TP (Verdadero Positivo):** El modelo dijo "venta alta" y lo era ✅
- **TN (Verdadero Negativo):** El modelo dijo "venta baja" y lo era ✅
- **FP (Falso Positivo):** El modelo dijo "venta alta" pero era baja ❌ (error Tipo I)
- **FN (Falso Negativo):** El modelo dijo "venta baja" pero era alta ❌ (error Tipo II)


---
## 5. Series de Tiempo — Pronóstico de Ventas con Prophet

### 🎯 Objetivo
Analizar la **evolución temporal de las ventas** y generar un **pronóstico** de las próximas semanas.

### ¿Qué es una Serie de Tiempo?
Una secuencia de observaciones ordenadas en el tiempo. En este caso: ventas diarias agregadas.

### Flujo de trabajo
```
Spark (agrupación diaria) → Pandas (Prophet) → Pronóstico → Visualización
```

> 📌 **¿Por qué salimos de Spark para Prophet?**  
> Prophet es una librería Python de un solo nodo. Usamos Spark para la agregación distribuida de datos y luego convertimos a Pandas solo para el modelo de pronóstico, que ya tiene un tamaño manejable.


In [0]:
# ── PASO 1: Agregar ventas diarias con Spark ─────────────────
# Esta operación puede procesar millones de registros distribuidos
ventas_diarias = (
    df.groupBy("Order Date")
    .agg(spark_sum("Sales").alias("DailySales"))
    .orderBy("Order Date")
)

print(f"📅 Días únicos con ventas: {ventas_diarias.count():,}")
display(ventas_diarias.limit(5))


In [0]:
# ── PASO 2: Convertir a Pandas y preparar para Prophet ───────
# Prophet requiere columnas específicas: 'ds' (fecha) y 'y' (valor)
pdf_ts = ventas_diarias.toPandas()
pdf_ts = pdf_ts.rename(columns={"Order Date": "ds", "DailySales": "y"})
pdf_ts = pdf_ts.sort_values("ds").reset_index(drop=True)
pdf_ts["ds"] = pd.to_datetime(pdf_ts["ds"])

print(f"📊 Rango: {pdf_ts['ds'].min().date()} → {pdf_ts['ds'].max().date()}")
print(f"   Total de días: {len(pdf_ts)}")

# Visualización de la serie original
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(pdf_ts["ds"], pdf_ts["y"], linewidth=0.8, color="#4e79a7", alpha=0.7)

# Media móvil 30 días
rolling = pdf_ts["y"].rolling(30, center=True).mean()
ax.plot(pdf_ts["ds"], rolling, color="#e15759", linewidth=2, label="Media Móvil 30 días")

ax.set_title("Ventas Diarias — Superstore", fontsize=13, fontweight="bold")
ax.set_xlabel("Fecha")
ax.set_ylabel("Ventas ($)")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend()
plt.tight_layout()
plt.show()


In [0]:
# ── PASO 3: Análisis de estacionalidad mensual ───────────────
pdf_ts["mes"] = pdf_ts["ds"].dt.month
pdf_ts["anio"] = pdf_ts["ds"].dt.year

ventas_mensuales = pdf_ts.groupby(["anio", "mes"])["y"].sum().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Ventas totales por mes (todos los años)
mensual_promedio = pdf_ts.groupby("mes")["y"].mean()
meses = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]
axes[0].bar(range(1,13), mensual_promedio.values, color="#76b7b2")
axes[0].set_xticks(range(1,13))
axes[0].set_xticklabels(meses, rotation=45)
axes[0].set_title("Venta Diaria Promedio por Mes", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Ventas promedio ($)")
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# Evolución por año
for anio in sorted(ventas_mensuales["anio"].unique()):
    subset = ventas_mensuales[ventas_mensuales["anio"] == anio]
    axes[1].plot(subset["mes"], subset["y"]/1e3, marker="o", label=str(anio))
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(meses, rotation=45)
axes[1].set_title("Ventas Mensuales por Año (miles $)", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Miles USD")
axes[1].legend()
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x:,.0f}K"))

plt.tight_layout()
plt.show()


### 📈 Modelo Prophet

**Prophet** (desarrollado por Meta/Facebook) es ideal para series de tiempo de negocios porque:
- Maneja automáticamente **estacionalidades** (semanal, mensual, anual)
- Es robusto a **datos faltantes** y **valores atípicos**
- Genera **intervalos de confianza** de manera automática
- Permite incorporar **festivos** y **eventos especiales**

#### Componentes del modelo:
$$y(t) = \text{tendencia}(t) + \text{estacionalidad}(t) + \text{festivos}(t) + \varepsilon$$


In [0]:
# ── Instalar Prophet si no está disponible ────────────────────
# En Databricks, ejecutar solo si es necesario:
#%pip install prophet

from prophet import Prophet
from prophet.plot import plot_plotly, plot_components_plotly

# ── PASO 4: Entrenar modelo Prophet ──────────────────────────
model_prophet = Prophet(
    yearly_seasonality=True,    # Captura patrones anuales
    weekly_seasonality=True,    # Captura patrones semanales
    daily_seasonality=False,    # No aplicable a datos diarios agregados
    changepoint_prior_scale=0.05,  # Flexibilidad de la tendencia
    interval_width=0.95         # Intervalo de confianza del 95%
)

# Entrenamos con todos los datos históricos
model_prophet.fit(pdf_ts[["ds","y"]])
print("✅ Modelo Prophet entrenado")


In [0]:
# ── PASO 5: Generar pronóstico ────────────────────────────────
DIAS_PRONOSTICO = 90  # Pronosticar 90 días hacia el futuro

# Crear dataframe de fechas futuras
futuro = model_prophet.make_future_dataframe(periods=DIAS_PRONOSTICO, freq="D")
pronostico = model_prophet.predict(futuro)

print(f"📅 Pronóstico generado hasta: {pronostico['ds'].max().date()}")
print("\nColumnas del pronóstico:")
print(pronostico[["ds","yhat","yhat_lower","yhat_upper"]].tail(5).to_string(index=False))


In [0]:
# ── PASO 6: Visualización del pronóstico ─────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

# Datos históricos
ax.scatter(pdf_ts["ds"], pdf_ts["y"], s=5, color="#4e79a7", alpha=0.5, label="Datos reales")

# Pronóstico
hist_mask = pronostico["ds"] <= pdf_ts["ds"].max()
fut_mask  = pronostico["ds"] >  pdf_ts["ds"].max()

ax.plot(pronostico.loc[hist_mask, "ds"], pronostico.loc[hist_mask, "yhat"],
        color="#e15759", linewidth=1.5, label="Ajuste histórico")
ax.plot(pronostico.loc[fut_mask, "ds"], pronostico.loc[fut_mask, "yhat"],
        color="#f28e2b", linewidth=2, linestyle="--", label=f"Pronóstico {DIAS_PRONOSTICO} días")
ax.fill_between(pronostico.loc[fut_mask, "ds"],
                pronostico.loc[fut_mask, "yhat_lower"],
                pronostico.loc[fut_mask, "yhat_upper"],
                alpha=0.25, color="#f28e2b", label="Intervalo confianza 95%")

ax.axvline(pdf_ts["ds"].max(), color="gray", linestyle=":", linewidth=1.5)
ax.text(pdf_ts["ds"].max(), ax.get_ylim()[1]*0.95, " Fin histórico", color="gray", fontsize=9)

ax.set_title("Pronóstico de Ventas Diarias — Prophet", fontsize=13, fontweight="bold")
ax.set_xlabel("Fecha")
ax.set_ylabel("Ventas ($)")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend()
plt.tight_layout()
plt.show()


In [0]:
# ── PASO 7: Descomposición de componentes ────────────────────
fig2 = model_prophet.plot_components(pronostico)
plt.suptitle("Componentes del Modelo Prophet", y=1.02, fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


### 🔍 Interpretación de las componentes

| Componente | Qué muestra |
|------------|------------|
| **Tendencia** | Dirección general de las ventas a lo largo del tiempo (¿crecen o decrecen?) |
| **Estacionalidad semanal** | ¿En qué días de la semana se vende más? |
| **Estacionalidad anual** | ¿En qué épocas del año se vende más? (Ej: fin de año) |

> 💡 Si la tendencia es creciente y el modelo captura bien la estacionalidad, los pronósticos serán más confiables.


---
## 6. Resumen y Comparación de Técnicas

| Técnica | Tipo de problema | Variable Y | Métricas clave |
|---------|-----------------|-----------|----------------|
| **Regresión Lineal** | Predicción numérica | Continua (Profit) | R², RMSE, MAE |
| **Regresión Logística** | Clasificación binaria | Categórica (HighSales 0/1) | AUC-ROC, Accuracy, F1 |
| **Series de Tiempo** | Pronóstico temporal | Ventas futuras | MAE, MAPE, cobertura IC |

### 🏗️ Ventajas de usar Spark para Big Data

1. **Escalabilidad:** El mismo código funciona con 10K o 10M registros
2. **Procesamiento distribuido:** Las transformaciones se ejecutan en paralelo
3. **Integración:** Spark MLlib se conecta directamente con el data lake
4. **Tolerancia a fallos:** Si un nodo falla, el trabajo continúa en otro

### 🚀 Próximos pasos sugeridos

- **Random Forest / Gradient Boosting** para mejorar la regresión y clasificación
- **K-Means Clustering** para segmentar clientes o productos
- **MLflow** para registrar y versionar experimentos
- **Feature Store** para reutilizar variables en múltiples modelos


In [0]:
# ── Resumen final de métricas ─────────────────────────────────
print("=" * 50)
print("  RESUMEN DE MODELOS — SUPERSTORE")
print("=" * 50)
print(f"  Regresión Lineal")
print(f"    R²   : {r2_score(y_test, y_pred):.4f}")
print(f"    RMSE : {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print("-" * 50)
print(f"  Regresión Logística")
print(f"    AUC-ROC  : {auc:.4f}")
print(f"    Accuracy : {acc:.4f}")
print(f"    F1 Score : {f1:.4f}")
print("-" * 50)
print(f"  Serie de Tiempo (Prophet)")
print(f"    Componentes: Tendencia + Estacionalidad semanal + anual")
print("=" * 50)